In [1]:
import pandas as pd
import numpy as np

from pybaseball import statcast
from pybaseball import playerid_reverse_lookup

In [2]:
df_player = pd.read_csv("player_dataset_2024.csv")
df_player

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,batter,pitcher,events,description,spin_dir,...,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,batter_name,pitcher_name
0,KC,2024-10-30,77.5,-1.11,5.65,657077,621111,strikeout,swinging_strike_blocked,NaN,...,-1.08,1.08,53.2,25.834292,-25.475081,32.831211,34.481443,57.315365,"Verdugo, Alex","Walker, Buehler"
1,KC,2024-10-30,78.7,-1.01,5.73,657077,621111,NaN,swinging_strike,NaN,...,-1.05,1.05,54.2,35.519261,-41.027263,35.112657,29.474286,57.624781,"Verdugo, Alex","Walker, Buehler"
2,FC,2024-10-30,93.1,-1.19,5.53,657077,621111,NaN,swinging_strike,NaN,...,-0.53,0.53,44.8,19.401316,-32.989729,26.710550,16.321290,37.919472,"Verdugo, Alex","Walker, Buehler"
3,KC,2024-10-30,78.5,-1.19,5.70,657077,621111,NaN,ball,NaN,...,-1.05,1.05,51.9,NaN,NaN,NaN,NaN,NaN,"Verdugo, Alex","Walker, Buehler"
4,KC,2024-10-30,77.4,-1.23,5.78,669224,621111,strikeout,swinging_strike,NaN,...,-1.08,1.08,50.0,22.643800,-12.035832,32.683497,40.578398,40.302028,"Wells, Austin","Walker, Buehler"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
752464,NaN,2024-03-15,NaN,NaN,NaN,666310,663903,NaN,swinging_strike,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Naylor, Bo","Brady, Singer"
752465,NaN,2024-03-15,NaN,NaN,NaN,666310,663903,NaN,swinging_strike,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Naylor, Bo","Brady, Singer"
752466,NaN,2024-03-15,NaN,NaN,NaN,608070,663903,double,hit_into_play,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ramírez, José","Brady, Singer"
752467,NaN,2024-03-15,NaN,NaN,NaN,665926,663903,field_out,hit_into_play,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Giménez, Andrés","Brady, Singer"


## The strongest batter by contact type
<br>Which Batter has the most strength? I am assuming that the person with the most barrels throughout the game has the most strength

In [3]:
#I mapped the numbers in the launch_speed_angle column to their relative contact type. I got this map from the Statcast documentation. 
#I then grouped the batters by the  number of barrels and Flares they hit.
b_strength_table = df_player[['batter_name', 'launch_speed_angle']]
b_strength_table['launch_speed_cat'] = b_strength_table['launch_speed_angle'].case_when([
    (b_strength_table['launch_speed_angle'] == 1, 'Weak'),
    (b_strength_table['launch_speed_angle'] == 2, 'Topped'),
    (b_strength_table['launch_speed_angle'] == 3, 'Under'),
    (b_strength_table['launch_speed_angle'] == 4, 'Flare/Burner'),
    (b_strength_table['launch_speed_angle'] == 5, 'Solid Contact'),
    (b_strength_table['launch_speed_angle'] == 6, 'Barrel'),
])
b_strength_table

C:\Users\skapuganti1\AppData\Local\Temp\ipykernel_4416\1979146479.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b_strength_table['launch_speed_cat'] = b_strength_table['launch_speed_angle'].case_when([


,batter_name,launch_speed_angle,launch_speed_cat
0,"Verdugo, Alex",NaN,NaN
1,"Verdugo, Alex",NaN,NaN
2,"Verdugo, Alex",NaN,NaN
3,"Verdugo, Alex",NaN,NaN
4,"Wells, Austin",NaN,NaN
...,...,...,...
752464,"Naylor, Bo",NaN,NaN
752465,"Naylor, Bo",NaN,NaN
752466,"Ramírez, José",NaN,NaN
752467,"Giménez, Andrés",NaN,NaN


In [4]:
b_strength = pd.pivot_table(b_strength_table, values = 'launch_speed_angle', index = 'batter_name', columns = 'launch_speed_cat', aggfunc = 'count')
b_strength

launch_speed_cat,Barrel,Flare/Burner,Solid Contact,Topped,Under,Weak
batter_name,,,,,,
"Abrams, Cj",29.0,102.0,40.0,117.0,111.0,18.0
"Abreu, José",2.0,17.0,4.0,40.0,26.0,2.0
"Abreu, Wilyer",31.0,67.0,21.0,71.0,91.0,4.0
"Acuña, Luisangel",3.0,5.0,1.0,12.0,10.0,2.0
"Acuña, Ronald",13.0,34.0,10.0,52.0,26.0,5.0
...,...,...,...,...,...,...
"Young, Jared",NaN,NaN,NaN,NaN,1.0,1.0
"Zavala, Seby",1.0,4.0,6.0,7.0,5.0,2.0
"Zimmer, Bradley",NaN,NaN,NaN,2.0,NaN,NaN


In [5]:
b_strength_max_barrel = b_strength['Barrel'].idxmax()
b_strength_max_flareburner = b_strength['Flare/Burner'].idxmax()
print("Maximum Barrels in the 2024 Season: ", b_strength_max_barrel, " with ",b_strength['Barrel'].max(), " Barrels!")
print("Maximum Flare/Burners in the 2024 Season: ", b_strength_max_flareburner, "with ",b_strength['Flare/Burner'].max(), "Flares/Burners.")

Maximum Barrels in the 2024 Season:  Ohtani, Shohei  with  112.0  Barrels!
Maximum Flare/Burners in the 2024 Season:  Arráez, Luis with  210.0 Flares/Burners.


In [6]:
strength_max_summary = pd.DataFrame({
    'Category' : ['Barrel', 'Flare/Burner'],
    'Player' : [b_strength_max_barrel, b_strength_max_flareburner],
    'Count': [b_strength['Barrel'].max(), b_strength['Flare/Burner'].max()]
})
strength_max_summary

,Category,Player,Count
0,Barrel,"Ohtani, Shohei",112.0
1,Flare/Burner,"Arráez, Luis",210.0


## The pitches preferred by pitchers

For each of the pitchers, which pitch type have they used the most in the 2024 MLB season?

In [36]:
#I created a pivot table with pitcher vs. pitch name. 
#Whichever pitch type the pitcher used the most(highest count), I assumed that to be their preferred pitch type.
pitch_table = df_player.groupby(['pitcher_name', 'pitch_name'])['pitch_name'].count().reset_index(name = 'pitch_type_count')
pitch_table = pitch_table.pivot(index = 'pitcher_name', columns = 'pitch_name', values = 'pitch_type_count')
pitch_table

pitch_name,4-Seam Fastball,Changeup,Curveball,Cutter,Eephus,Forkball,Knuckle Curve,Knuckleball,Other,Pitch Out,Screwball,Sinker,Slider,Slow Curve,Slurve,Split-Finger,Sweeper
pitcher_name,,,,,,,,,,,,,,,,,
"A. j., Minter",259.0,95.0,NaN,186.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"A. j., Puk",507.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,173.0,254.0,NaN,NaN,30.0,192.0
"Aaron, Ashby",13.0,119.0,104.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,255.0,89.0,NaN,NaN,NaN,NaN
"Aaron, Brooks",112.0,54.0,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,99.0,131.0,NaN,NaN,NaN,NaN
"Aaron, Bummer",99.0,11.0,22.0,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,396.0,NaN,NaN,NaN,NaN,425.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Zack, Kelly",286.0,200.0,NaN,209.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,260.0
"Zack, Littell",574.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,380.0,895.0,NaN,NaN,586.0,129.0
"Zack, Thompson",139.0,NaN,59.0,NaN,NaN,79.0,NaN,NaN,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN


In [8]:
pitch_4seam_max = pitch_table['4-Seam Fastball'].idxmax()
pitch_changeup_max = pitch_table['Changeup'].idxmax()
pitch_curveball_max = pitch_table['Curveball'].idxmax()
pitch_cutter_max = pitch_table['Cutter'].idxmax()
pitch_eephus_max = pitch_table['Eephus'].idxmax()
pitch_forkball_max = pitch_table['Forkball'].idxmax()
pitch_knuckleball_max = pitch_table['Knuckleball'].idxmax()
pitch_other_max = pitch_table['Other'].idxmax()
pitch_pitchout_max = pitch_table['Pitch Out'].idxmax()
pitch_screwball_max = pitch_table['Screwball'].idxmax()
pitch_sinker_max = pitch_table['Sinker'].idxmax()
pitch_slider_max = pitch_table['Slider'].idxmax()
pitch_slowcurve_max = pitch_table['Slow Curve'].idxmax()
pitch_slurve_max = pitch_table['Slurve'].idxmax()
pitch_splitfinger_max = pitch_table['Split-Finger'].idxmax()
pitch_sweeper_max = pitch_table['Sweeper'].idxmax()

print("Maximum 4-Seam Fastballs in the 2024 Season: ", pitch_4seam_max, " with ",pitch_table['4-Seam Fastball'].max(), " fastballs!")
print("Maximum Changeups in the 2024 Season: ", pitch_changeup_max, " with ",pitch_table['Changeup'].max(), " Changeups!")
print("Maximum Curveballs in the 2024 Season: ", pitch_curveball_max, " with ",pitch_table['Curveball'].max(), " Curveballs!")
print("Maximum Cutters in the 2024 Season: ", pitch_cutter_max, " with ",pitch_table['Cutter'].max(), " Cutters!")
print("Maximum Eephuses in the 2024 Season: ", pitch_eephus_max, " with ",pitch_table['Eephus'].max(), " Eephuses!")
print("Maximum Forkballs in the 2024 Season: ", pitch_forkball_max, " with ",pitch_table['Forkball'].max(), " Forkballs!")
print("Maximum Knuckleballs in the 2024 Season: ", pitch_knuckleball_max, " with ",pitch_table['Knuckleball'].max(), " Knuckleballs!")
print("Maximum Pitch outs in the 2024 Season: ", pitch_pitchout_max, " with ",pitch_table['Pitch Out'].max(), " Pitch Outs!")
print("Maximum Screwballs in the 2024 Season: ", pitch_screwball_max, " with ",pitch_table['Screwball'].max(), " Screwballs!")
print("Maximum Sinkers in the 2024 Season: ", pitch_sinker_max, " with ",pitch_table['Sinker'].max(), " Sinkers!")
print("Maximum Sliders in the 2024 Season: ", pitch_slider_max, " with ",pitch_table['Slider'].max(), " Sliders!")
print("Maximum Slow Curves in the 2024 Season: ", pitch_slowcurve_max, " with ",pitch_table['Slow Curve'].max(), " Slow Curve!")
print("Maximum 4-Seam Slurves in the 2024 Season: ", pitch_slurve_max, " with ",pitch_table['Slurve'].max(), " Slurves!")
print("Maximum Split-Fingers in the 2024 Season: ", pitch_splitfinger_max, " with ",pitch_table['Split-Finger'].max(), " Split-Fingers!")
print("Maximum 4-Seam Sweepers in the 2024 Season: ", pitch_sweeper_max, " with ",pitch_table['Sweeper'].max(), " Sweepers!")
print("Other Types of pitches thrown in the 2024 Season: ", pitch_other_max, " with ",pitch_table['Other'].max(), " other types of pitches.")

Maximum 4-Seam Fastballs in the 2024 Season:  Carlos, Rodón  with  1790.0  fastballs!
Maximum Changeups in the 2024 Season:  Tyler, Anderson  with  1082.0  Changeups!
Maximum Curveballs in the 2024 Season:  Charlie, Morton  with  1197.0  Curveballs!
Maximum Cutters in the 2024 Season:  Corbin, Burnes  with  1397.0  Cutters!
Maximum Eephuses in the 2024 Season:  Garrett, Stubbs  with  52.0  Eephuses!
Maximum Forkballs in the 2024 Season:  Zack, Thompson  with  79.0  Forkballs!
Maximum Knuckleballs in the 2024 Season:  Matt, Waldron  with  936.0  Knuckleballs!
Maximum Pitch outs in the 2024 Season:  Andre, Pallante  with  3.0  Pitch Outs!
Maximum Screwballs in the 2024 Season:  Brent, Honeywell  with  168.0  Screwballs!
Maximum Sinkers in the 2024 Season:  Cristopher, Sánchez  with  1403.0  Sinkers!
Maximum Sliders in the 2024 Season:  Dylan, Cease  with  1446.0  Sliders!
Maximum Slow Curves in the 2024 Season:  Miles, Mikolas  with  15.0  Slow Curve!
Maximum 4-Seam Slurves in the 2024 S

## Batter/Pitcher preferences

Which side of the home plate does a batter prefer? How many of them prefer the right side or the left side? Similarly, how many pitchers prefer their right vs. their left hand?

In [37]:
#I did a simple unique count - to see how many batters preferred the right side of the homeplate to the left. The right side was preferred more often.
batter_side = df_player.groupby(['stand'])['batter_name'].nunique()
print("Number of Batters who prefer their left vs. right side of the plate: ")
batter_side

Number of Batters who prefer their left vs. right side of the plate: 


stand
L    366
R    547
Name: batter_name, dtype: int64

In [38]:
#Looks like the right-handed batters were more active! There are also more right-handed batters anyway...
df_player['stand'].value_counts()

stand
R    428653
L    323816
Name: count, dtype: int64

In [10]:
#Performing a unique count of players who prefer which hand as their throwing hand...
pitcher_side = df_player.groupby(['p_throws'])['pitcher_name'].nunique()
print("Number of pitchers who prefer throwing with their right hand vs. left: ")
pitcher_side

Number of pitchers who prefer throwing with their right hand vs. left: 


p_throws
L    244
R    711
Name: pitcher_name, dtype: int64

## What events occur when a batter and a pitcher are paired together?

Based on the batters' positioning with respect to the home plate and the pitchers' preferred throwing hand, what events generally occur most frequently? The second part of the analysis focuses on only the 'interesting events' such as a strikeout, a homerun, singles, doubles, and triples... 

In [12]:
#I grouped the data by the batter's preference, the pitchers preference, and counted the events that occurred.
event_table = df_player.groupby(['stand', 'p_throws', 'events'])['events'].count().unstack(fill_value = 0).reset_index()
event_table

events,stand,p_throws,catcher_interf,double,double_play,field_error,field_out,fielders_choice,fielders_choice_out,force_out,...,sac_bunt,sac_fly,sac_fly_double_play,single,strikeout,strikeout_double_play,triple,triple_play,truncated_pa,walk
0,L,L,10,574,29,76,6048,37,24,337,...,56,96,1,2271,3636,6,73,0,29,1127
1,L,R,52,2905,140,341,27842,128,86,958,...,141,442,4,9330,14939,41,339,0,127,6103
2,R,L,13,1723,62,276,14784,76,67,844,...,102,263,5,5369,8181,17,117,0,51,3048
3,R,R,33,3130,131,517,30016,177,146,1577,...,173,534,5,10771,17281,53,225,2,137,5226


In [13]:
print(event_table.columns)

Index(['stand', 'p_throws', 'catcher_interf', 'double', 'double_play',
       'field_error', 'field_out', 'fielders_choice', 'fielders_choice_out',
       'force_out', 'grounded_into_double_play', 'hit_by_pitch', 'home_run',
       'intent_walk', 'sac_bunt', 'sac_fly', 'sac_fly_double_play', 'single',
       'strikeout', 'strikeout_double_play', 'triple', 'triple_play',
       'truncated_pa', 'walk'],
      dtype='object', name='events')


In [14]:
event_cols = event_table.columns[2:]
event_table[event_cols] = event_table[event_cols].apply(pd.to_numeric, errors = 'coerce').fillna(0)
event_table['dominant_event'] = event_table[event_cols].idxmax(axis = 1)
event_table['dominant_event_count'] = event_table[event_cols].max(axis = 1)
event_result = event_table[['stand', 'p_throws', 'dominant_event', 'dominant_event_count']]
print(event_result)

events stand p_throws dominant_event  dominant_event_count
0          L        L      field_out                  6048
1          L        R      field_out                 27842
2          R        L      field_out                 14784
3          R        R      field_out                 30016


In [15]:
#Looking at the key events such as the home runs, strikeouts, singles, doubles, and triples
# Also, excluding the dominating event - field outs
interesting_event_table = event_table[['stand', 'p_throws', 'home_run', 'single', 'double', 'triple', 'strikeout']]
interesting_event_table_cols = ['home_run', 'single', 'double', 'triple', 'strikeout']

interesting_event_table['dominant_event'] = interesting_event_table[interesting_event_table_cols].idxmax(axis = 1)
interesting_event_table['dominant_event_count'] = interesting_event_table[interesting_event_table_cols].max(axis = 1)

interesting_event_table['total_events'] = interesting_event_table[interesting_event_table_cols].sum(axis = 1)
interesting_event_table_pct = (interesting_event_table[interesting_event_table_cols]
                               .div(interesting_event_table['total_events'], axis= 0)
                               .mul(100)
                               .round(2))
interesting_event_result = interesting_event_table[['stand', 'p_throws', 'dominant_event', 'dominant_event_count']]
print(interesting_event_result)

events stand p_throws dominant_event  dominant_event_count
0          L        L      strikeout                  3636
1          L        R      strikeout                 14939
2          R        L      strikeout                  8181
3          R        R      strikeout                 17281


C:\Users\skapuganti1\AppData\Local\Temp\ipykernel_4416\330981699.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interesting_event_table['dominant_event'] = interesting_event_table[interesting_event_table_cols].idxmax(axis = 1)
C:\Users\skapuganti1\AppData\Local\Temp\ipykernel_4416\330981699.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interesting_event_table['dominant_event_count'] = interesting_event_table[interesting_event_table_cols].max(axis = 1)
C:\Users\skapuganti1\AppData\Local\Temp\ipyke

In [16]:
#Looks like for all the combinations, strikeouts had the highest frequency. However, the L-L combination had the highest percentage of strikeouts...
interesting_event_table_pct = pd.concat([interesting_event_table[['stand', 'p_throws']], interesting_event_table_pct], axis = 1)
interesting_event_table_pct

events,stand,p_throws,home_run,single,double,triple,strikeout
0,L,L,5.10,32.88,8.31,1.06,52.65
1,L,R,7.30,31.43,9.79,1.14,50.33
2,R,L,7.01,32.44,10.41,0.71,49.43
3,R,R,6.47,32.08,9.32,0.67,51.47


## Launch speed vs. Zone
Trying to find a pattern between the contact type and the zone where the ball lands. The mapping is from the statcast documentation. The results are also normalized to find a more accurate, unbiased relationship, rather than the event frequency affecting the results. 

In [17]:
#Mapping the launch_speed_angle to the respective categories from the documentation. 
#Then, grouping the data based on the category to count the number of times the ball fell in a particular zone
ball_landing = df_player[['launch_speed_angle', 'zone']]
ball_landing['launch_speed_cat'] = ball_landing['launch_speed_angle'].case_when([
    (ball_landing['launch_speed_angle'] == 1, 'Weak'),
    (ball_landing['launch_speed_angle'] == 2, 'Topped'),
    (ball_landing['launch_speed_angle'] == 3, 'Under'),
    (ball_landing['launch_speed_angle'] == 4, 'Flare/Burner'),
    (ball_landing['launch_speed_angle'] == 5, 'Solid Contact'),
    (ball_landing['launch_speed_angle'] == 6, 'Barrel'),
])
x = ball_landing.dropna(subset = ['launch_speed_angle'])
x

C:\Users\skapuganti1\AppData\Local\Temp\ipykernel_4416\1366473643.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ball_landing['launch_speed_cat'] = ball_landing['launch_speed_angle'].case_when([


,launch_speed_angle,zone,launch_speed_cat
11,2.0,2.0,Topped
16,4.0,13.0,Flare/Burner
28,6.0,4.0,Barrel
29,4.0,13.0,Flare/Burner
36,3.0,6.0,Under
...,...,...,...
752271,6.0,8.0,Barrel
752277,4.0,9.0,Flare/Burner
752281,4.0,8.0,Flare/Burner
752296,3.0,8.0,Under


In [18]:
ball_landing_pivot = (ball_landing
                      .groupby(['launch_speed_cat', 'zone'])
                      .size()
                      .unstack(fill_value = 0))
ball_landing_pivot['Total'] = ball_landing_pivot.sum(axis = 1)
ball_landing_pivot

zone,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,11.0,12.0,13.0,14.0,Total
launch_speed_cat,,,,,,,,,,,,,,
Barrel,443,1035,468,1147,2579,1196,592,1475,594,149,136,171,158,10143
Flare/Burner,1463,2097,1291,3561,4987,3397,2411,3919,2677,928,943,1657,1976,31307
Solid Contact,366,763,419,927,1719,943,518,1217,543,150,181,168,166,8080
Topped,1443,1807,1249,4254,4988,3771,3649,5053,3980,1331,1158,3320,3902,39905
Under,2289,3469,2159,3573,5515,3855,1815,2984,2194,1516,1448,1243,1496,33556
Weak,325,197,295,468,378,434,276,356,381,830,529,550,698,5717


In [19]:
ball_landing_pct = (ball_landing_pivot.div(ball_landing_pivot['Total'], axis = 0)*100).round(2)
ball_landing_pct = ball_landing_pct.drop(columns = ['Total'])
ball_landing_pct

zone,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,11.0,12.0,13.0,14.0
launch_speed_cat,,,,,,,,,,,,,
Barrel,4.37,10.20,4.61,11.31,25.43,11.79,5.84,14.54,5.86,1.47,1.34,1.69,1.56
Flare/Burner,4.67,6.70,4.12,11.37,15.93,10.85,7.70,12.52,8.55,2.96,3.01,5.29,6.31
Solid Contact,4.53,9.44,5.19,11.47,21.27,11.67,6.41,15.06,6.72,1.86,2.24,2.08,2.05
Topped,3.62,4.53,3.13,10.66,12.50,9.45,9.14,12.66,9.97,3.34,2.90,8.32,9.78
Under,6.82,10.34,6.43,10.65,16.44,11.49,5.41,8.89,6.54,4.52,4.32,3.70,4.46
Weak,5.68,3.45,5.16,8.19,6.61,7.59,4.83,6.23,6.66,14.52,9.25,9.62,12.21


In [20]:
#Looks like most of the types of hits fall in the inner matrix (1-9), mainly zone 5, except weak shots that fall into zone 11
ball_landing_pct['max_zone'] = ball_landing_pct.idxmax(axis = 1)
ball_landing_pct['max_zone_cnt'] = ball_landing_pct.max(axis = 1)
print(ball_landing_pct[['max_zone', 'max_zone_cnt']])

zone             max_zone max_zone_cnt
launch_speed_cat                      
Barrel                5.0        25.43
Flare/Burner          5.0        15.93
Solid Contact         5.0        21.27
Topped                8.0        12.66
Under                 5.0        16.44
Weak                 11.0        14.52


## Which batter has hit the highest number of home runs, and which pitcher recorded the highest number of strikeouts?

A simple group-by analysis to see the strongest batters and pitchers in the 2024 MLB season

In [21]:
#How many batters hit a home run, and who hit the highest number of home runs?
home_runs = df_player[df_player['events']=="home_run"]
home_runs = home_runs[['batter_name', 'events']]
home_runs

,batter_name,events
254,"Stanton, Giancarlo",home_run
321,"Chisholm, Jazz",home_run
325,"Judge, Aaron",home_run
379,"Torres, Gleyber",home_run
466,"Wells, Austin",home_run
...,...,...
751945,"Fermín, José",home_run
752025,"Berti, Jon",home_run
752085,"Arcia, Orlando",home_run
752241,"Malloy, Justyn-henry",home_run


In [22]:
#557 batters hit a home run, and the highest was hit by Aaron Judge (61 home runs) in the 2024 season
home_runs_batter = home_runs['batter_name'].value_counts().reset_index()
home_runs_batter.sort_values(by = 'count', ascending = False)

,batter_name,count
0,"Judge, Aaron",61
1,"Ohtani, Shohei",57
2,"Soto, Juan",45
3,"Santander, Anthony",45
4,"Ramírez, José",43
...,...,...
503,"Contreras, Mark",1
502,"Devanney, Cam",1
501,"Shaw, Matt",1
500,"Hudson, Joe",1


In [23]:
#How many pitchers threw a strikeout, and who threw the highest number of strikeouts?
pitch_strikeout = df_player[df_player['events']=="strikeout"]
pitch_strikeout = pitch_strikeout[['pitcher_name', 'events']]
pitch_strikeout

,pitcher_name,events
0,"Walker, Buehler",strikeout
4,"Walker, Buehler",strikeout
17,"Mark, Leiter",strikeout
32,"Blake, Treinen",strikeout
53,"Luke, Weaver",strikeout
...,...,...
752447,"Tyler, Beede",strikeout
752452,"Brady, Singer",strikeout
752455,"Brady, Singer",strikeout
752458,"Tyler, Beede",strikeout


In [24]:
#862 pitchers threw a strikeout, and the highest was thrown by Tarik Skubal (258 strikeouts) in the 2024 season
pitch_strikeout_res = pitch_strikeout['pitcher_name'].value_counts().reset_index()
pitch_strikeout_res.sort_values(by = 'count', ascending = False)

,pitcher_name,count
0,"Tarik, Skubal",258
1,"Cole, Ragans",247
2,"Zack, Wheeler",238
3,"Dylan, Cease",234
4,"Logan, Gilbert",231
...,...,...
834,"Cam, Sanders",1
835,"Evan, Kravetz",1
836,"Ricky, Vanasco",1
837,"Ben, Rortvedt",1


## Which team is the strongest, with the most barrels?
I assumed the team that hit the most barrels had the strongest batting team.

In [25]:
#Barrels by hometeam - filtering the main table to get the teams who hit barrels
hometeam_barrels = df_player[df_player['launch_speed_angle']==6]
hometeam_barrels = hometeam_barrels[['home_team', 'launch_speed_angle']]
hometeam_barrels

,home_team,launch_speed_angle
28,NYY,6.0
122,NYY,6.0
182,NYY,6.0
235,NYY,6.0
254,NYY,6.0
...,...,...
751972,STL,6.0
751994,STL,6.0
752004,STL,6.0
752085,DET,6.0


In [26]:
#The New York Yankees hit the most number of Barrels in the 2024 season
hometeam_barrels_res = hometeam_barrels['home_team'].value_counts().reset_index()
hometeam_barrels_res.sort_values(by = 'count', ascending = False)

,home_team,count
0,NYY,448
1,BAL,402
2,LAD,391
3,NYM,381
4,MIA,376
5,AZ,376
6,PIT,369
7,SD,364
8,COL,363
9,MIN,360


## Based on the release speed, what events occurred?

Determining a pattern between the pitcher's release speed and the event that subsequently occurs

In [27]:
#Bucketing the release speed to give cleaner results
release_speed_data = df_player[['release_speed', 'events']]
release_speed_data = release_speed_data.dropna(subset = ['release_speed'])
release_speed_data['release_speed_cat'] = release_speed_data['release_speed'].case_when([
    ((release_speed_data['release_speed']>=30) & (release_speed_data['release_speed']<40), '30-40'),
    ((release_speed_data['release_speed']>=40) & (release_speed_data['release_speed']<50), '40-50'),
    ((release_speed_data['release_speed']>=50) & (release_speed_data['release_speed']<60), '50-60'),
    ((release_speed_data['release_speed']>=60) & (release_speed_data['release_speed']<70), '60-70'),
    ((release_speed_data['release_speed']>=70) & (release_speed_data['release_speed']<80), '70-80'),
    ((release_speed_data['release_speed']>=80) & (release_speed_data['release_speed']<90), '80-90'),
    ((release_speed_data['release_speed']>=90) & (release_speed_data['release_speed']<100), '90-100'),
    (release_speed_data['release_speed']>=100,'100+')
])
    
release_speed_data

,release_speed,events,release_speed_cat
0,77.5,strikeout,70-80
1,78.7,NaN,70-80
2,93.1,NaN,90-100
3,78.5,NaN,70-80
4,77.4,strikeout,70-80
...,...,...,...
752300,92.5,NaN,90-100
752301,83.5,single,80-90
752302,83.6,NaN,80-90
752303,93.8,NaN,90-100


In [28]:
# Grouping the data to create a table to see which events occurred most based on the pitcher's release speed
release_speed_table = (release_speed_data
                       .groupby(['release_speed_cat', 'events'])
                       .size()
                       .unstack(fill_value = 0))
release_speed_table

events,catcher_interf,double,double_play,field_error,field_out,fielders_choice,fielders_choice_out,force_out,grounded_into_double_play,hit_by_pitch,...,sac_bunt,sac_fly,sac_fly_double_play,single,strikeout,strikeout_double_play,triple,triple_play,truncated_pa,walk
release_speed_cat,,,,,,,,,,,,,,,,,,,,,
100+,0,23,2,3,299,2,2,17,9,8,...,1,5,0,136,348,0,1,0,2,80
30-40,0,1,0,0,15,0,0,0,2,1,...,0,0,0,6,0,0,0,0,0,1
40-50,0,5,0,1,48,0,0,1,2,2,...,0,1,0,17,0,0,1,0,0,5
50-60,0,16,0,0,57,0,0,4,2,3,...,0,1,0,27,0,0,0,0,0,12
60-70,0,7,0,1,84,0,0,4,6,7,...,0,4,0,25,16,0,1,0,0,14
70-80,4,546,22,78,4870,18,22,214,168,162,...,30,85,1,1508,2823,4,47,0,21,580
80-90,38,3457,166,504,32450,176,152,1699,1386,787,...,174,551,10,11036,20603,48,288,0,122,4894
90-100,63,3999,162,575,38871,209,140,1681,1757,1141,...,260,657,3,14174,18986,62,386,2,185,9429


In [29]:
#Normalizing the data to see unbiased results of the relation between the launch speed and the event occurring 
#So that the frequency of the event does not affect the results  
release_speed_table['total_events'] = release_speed_table.sum(axis = 1)
release_speed_table_pct = release_speed_table.div(release_speed_table['total_events'], axis = 0).mul(100).round(2)
release_speed_table_pct = release_speed_table_pct.drop(columns = ['total_events'])

release_speed_table_cols = release_speed_table_pct.columns.to_list()
release_speed_table_pct['dominant_event'] = release_speed_table_pct[release_speed_table_cols].idxmax(axis = 1)
release_speed_table_pct['dominant_event_count'] = release_speed_table_pct[release_speed_table_cols].max(axis = 1)
release_speed_table_pct

events,catcher_interf,double,double_play,field_error,field_out,fielders_choice,fielders_choice_out,force_out,grounded_into_double_play,hit_by_pitch,...,sac_fly_double_play,single,strikeout,strikeout_double_play,triple,triple_play,truncated_pa,walk,dominant_event,dominant_event_count
release_speed_cat,,,,,,,,,,,,,,,,,,,,,
100+,0.00,2.40,0.21,0.31,31.21,0.21,0.21,1.77,0.94,0.84,...,0.00,14.20,36.33,0.00,0.10,0.0,0.21,8.35,strikeout,36.33
30-40,0.00,3.70,0.00,0.00,55.56,0.00,0.00,0.00,7.41,3.70,...,0.00,22.22,0.00,0.00,0.00,0.0,0.00,3.70,field_out,55.56
40-50,0.00,5.81,0.00,1.16,55.81,0.00,0.00,1.16,2.33,2.33,...,0.00,19.77,0.00,0.00,1.16,0.0,0.00,5.81,field_out,55.81
50-60,0.00,12.90,0.00,0.00,45.97,0.00,0.00,3.23,1.61,2.42,...,0.00,21.77,0.00,0.00,0.00,0.0,0.00,9.68,field_out,45.97
60-70,0.00,3.87,0.00,0.55,46.41,0.00,0.00,2.21,3.31,3.87,...,0.00,13.81,8.84,0.00,0.55,0.0,0.00,7.73,field_out,46.41
70-80,0.03,4.71,0.19,0.67,42.04,0.16,0.19,1.85,1.45,1.40,...,0.01,13.02,24.37,0.03,0.41,0.0,0.18,5.01,field_out,42.04
80-90,0.05,4.27,0.21,0.62,40.09,0.22,0.19,2.10,1.71,0.97,...,0.01,13.64,25.46,0.06,0.36,0.0,0.15,6.05,field_out,40.09
90-100,0.07,4.18,0.17,0.60,40.66,0.22,0.15,1.76,1.84,1.19,...,0.00,14.82,19.86,0.06,0.40,0.0,0.19,9.86,field_out,40.66


In [30]:
#Looks like for all the release speeds, fieldouts happened the most often...
print(release_speed_table_pct[['dominant_event', 'dominant_event_count']])

events            dominant_event  dominant_event_count
release_speed_cat                                     
100+                   strikeout                 36.33
30-40                  field_out                 55.56
40-50                  field_out                 55.81
50-60                  field_out                 45.97
60-70                  field_out                 46.41
70-80                  field_out                 42.04
80-90                  field_out                 40.09
90-100                 field_out                 40.66


In [31]:
#To even out the playing field, excluding the fieldout events...
release_speed_table_minus_fieldout = release_speed_table.drop(columns = 'field_out')
release_speed_table_minus_fieldout
release_speed_table_minus_fieldout['total_events'] = release_speed_table_minus_fieldout.sum(axis = 1)
release_speed_table_minus_fieldout_pct = release_speed_table_minus_fieldout.div(release_speed_table_minus_fieldout['total_events'], axis = 0).mul(100).round(2)
release_speed_table_minus_fieldout_pct = release_speed_table_minus_fieldout_pct.drop(columns = ['total_events'])
release_speed_table_minus_fieldout_pct

events,catcher_interf,double,double_play,field_error,fielders_choice,fielders_choice_out,force_out,grounded_into_double_play,hit_by_pitch,home_run,sac_bunt,sac_fly,sac_fly_double_play,single,strikeout,strikeout_double_play,triple,triple_play,truncated_pa,walk
release_speed_cat,,,,,,,,,,,,,,,,,,,,
100+,0.00,1.42,0.12,0.19,0.12,0.12,1.05,0.56,0.49,1.24,0.06,0.31,0.00,8.41,21.52,0.00,0.06,0.0,0.12,4.95
30-40,0.00,2.56,0.00,0.00,0.00,0.00,0.00,5.13,2.56,2.56,0.00,0.00,0.00,15.38,0.00,0.00,0.00,0.0,0.00,2.56
40-50,0.00,4.03,0.00,0.81,0.00,0.00,0.81,1.61,1.61,2.42,0.00,0.81,0.00,13.71,0.00,0.00,0.81,0.0,0.00,4.03
50-60,0.00,8.38,0.00,0.00,0.00,0.00,2.09,1.05,1.57,1.05,0.00,0.52,0.00,14.14,0.00,0.00,0.00,0.0,0.00,6.28
60-70,0.00,2.52,0.00,0.36,0.00,0.00,1.44,2.16,2.52,4.32,0.00,1.44,0.00,8.99,5.76,0.00,0.36,0.0,0.00,5.04
70-80,0.02,2.98,0.12,0.43,0.10,0.12,1.17,0.92,0.89,2.08,0.16,0.46,0.01,8.24,15.43,0.02,0.26,0.0,0.11,3.17
80-90,0.03,2.67,0.13,0.39,0.14,0.12,1.31,1.07,0.61,1.85,0.13,0.43,0.01,8.53,15.92,0.04,0.22,0.0,0.09,3.78
90-100,0.04,2.62,0.11,0.38,0.14,0.09,1.10,1.15,0.75,1.88,0.17,0.43,0.00,9.30,12.46,0.04,0.25,0.0,0.12,6.19


In [32]:
#Here, looks like for the higher release speeds, strikeouts were more common, and for the lower speeds, singles occured most often... 
release_speed_table_minus_fieldout_pct_cols = release_speed_table_minus_fieldout_pct.columns.to_list()
release_speed_table_minus_fieldout_pct['dominant_event'] = release_speed_table_minus_fieldout_pct[release_speed_table_minus_fieldout_pct_cols].idxmax(axis = 1)
release_speed_table_minus_fieldout_pct['dominant_event_count'] = release_speed_table_minus_fieldout_pct[release_speed_table_minus_fieldout_pct_cols].max(axis = 1)
print(release_speed_table_minus_fieldout_pct[['dominant_event', 'dominant_event_count']])

events            dominant_event  dominant_event_count
release_speed_cat                                     
100+                   strikeout                 21.52
30-40                     single                 15.38
40-50                     single                 13.71
50-60                     single                 14.14
60-70                     single                  8.99
70-80                  strikeout                 15.43
80-90                  strikeout                 15.92
90-100                 strikeout                 12.46


## WOBA Analyses

I read online that wOBA is an important metric for measuring a batter's performance in baseball. The analysis focuses on two aspects - pitch type vs. wOBA score and the wOBA scores vs. the zone the ball falls in.

In [34]:
#Grouped the data by the pitch name, and looked at the records that are complete - woba_denom = 1
#Looks like Eephus pitches are easier for Batters to get a higher wOBA score, whereas split fingers usually result in lower wOBA scores...
pitch_woba_avg = df_player[df_player['woba_denom']== 1]
pitch_woba_avg= pitch_woba_avg.groupby('pitch_name')['woba_value'].mean().reset_index()
pitch_woba_avg = pitch_woba_avg.sort_values('woba_value', ascending = False)
pitch_woba_avg

,pitch_name,woba_value
4,Eephus,0.426904
8,Other,0.410250
10,Sinker,0.357407
9,Screwball,0.356410
3,Cutter,0.344454
0,4-Seam Fastball,0.338163
5,Forkball,0.323171
7,Knuckleball,0.301373
1,Changeup,0.295996
2,Curveball,0.292335


In [35]:
#Grouping the data by the zone, and taking the mean of the wOBA scores to see which zone has the highest wOBA score. 
#To find a relation between zones and wOBA scores, it looks like the highest wOBA scores occur when the ball falls in zone 5, followed by zone 11? 
zone_woba_avg = df_player[df_player['woba_denom']== 1]
zone_woba_avg= zone_woba_avg.groupby('zone')['woba_value'].mean().reset_index()
zone_woba_avg = zone_woba_avg.sort_values('woba_value', ascending = False)
zone_woba_avg

,zone,woba_value
4,5.0,0.401658
9,11.0,0.377281
7,8.0,0.351412
3,4.0,0.338404
5,6.0,0.327526
10,12.0,0.322566
1,2.0,0.316225
6,7.0,0.297774
8,9.0,0.276737
11,13.0,0.274545
